In [ ]:
import pandas as pd
import requests
import logging
import os
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    filename="script_cleaning_file.log")
telegram_bot=os.environ.get("TELEGRAM_BOT")
chat_id=os.environ.get("CHAT_ID")
url_doc=f"https://api.telegram.org/bot{telegram_bot}/sendDocument"
url_allert=f"https://api.telegram.org/bot{telegram_bot}/sendMessage"
try:
    df=pd.read_csv("outputs/tekworld_output.csv")
    washing=df[df["Category"]=="lavatrici"]
    washing=washing.dropna(axis=1, how="all")
    fridges=df[df["Category"]=="frigoriferi"]
    fridges=fridges.dropna(axis=1, how="all")
    dishes=df[df["Category"]=="lavastoviglie"]
    dishes=dishes.dropna(axis=1, how="all")
    with pd.ExcelWriter("outputs/final_output.xlsx") as writer:
        washing.to_excel(writer, sheet_name="WASHING MACHINES", index=False)
        fridges.to_excel(writer, sheet_name="FRIDGES", index=False)
        dishes.to_excel(writer, sheet_name="DISHWASHERS", index=False)
    with open("outputs/final_output.xlsx", "rb") as file:
        requests.post(url_doc, data={"chat_id": chat_id}, files={"document": file})
except FileNotFoundError as error:
    requests.get(url_allert, params={"chat_id": chat_id, "text": f"Error reading the CSV file: {error}"})